# Split and seed variance (Colab)

Every number in the write-up is one training run on one ingredient split. This notebook runs `scripts/12_split_seeds.py train` on a Colab GPU: the final configuration (SapBERT + ingredient negatives + strength normalizer) on six ingredient splits at seed 42, plus the published split at two more seeds. Eight runs, about two minutes each on a T4, logged to the `rxnorm-vandf` project under the run group `split-seeds`. No model artifact is logged.

The splits come from the W&B artifact `vandf-rxnorm-splits`, built and uploaded locally with `12_split_seeds.py build` and `upload` (the published dataset artifact `vandf-rxnorm-pairs` is untouched).

**Before running:** Runtime → Change runtime type → **T4 GPU**. Add the `WANDB_API_KEY` secret (key icon, *Notebook access* on).

## 1. Check the GPU
Expect a Tesla T4 (or better). If this prints nothing, the runtime type is still CPU.

In [ ]:
!nvidia-smi -L

## 2. Get the code
A plain clone of the public repo. `BRANCH` lets this run from a feature branch before it is merged; re-running the cell pulls the latest commit.

In [ ]:
import os, subprocess

REPO = "kvenanzi/rxnorm"
BRANCH = "main"
URL = f"https://github.com/{REPO}.git"
if not os.path.isdir("rxnorm"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL], check=True)
else:
    subprocess.run(["git", "-C", "rxnorm", "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", "rxnorm", "pull", "-q"], check=True)
!git -C rxnorm log --oneline -1

## 3. Make the package importable and install what Colab lacks
The clone directory goes on `sys.path`, so `import rxnorm_vandf` reads the code straight from the clone (a `git pull` in cell 2 is picked up immediately). An editable `pip install -e` would need a kernel restart to take effect in Colab.

The pip line adds only what Colab doesn't already ship, without upgrading what it does: upgrading Colab's numpy/pandas/torch inside a running kernel breaks its preinstalled stack.

In [ ]:
import sys
if os.path.abspath("rxnorm") not in sys.path:
    sys.path.insert(0, os.path.abspath("rxnorm"))
%pip install -q duckdb sentence-transformers datasets wandb accelerate
import torch, sentence_transformers, numpy, pandas, rxnorm_vandf
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "| sentence-transformers", sentence_transformers.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| rxnorm_vandf from", os.path.dirname(rxnorm_vandf.__file__))

## 4. Log in to Weights & Biases
The API key comes from Colab Secrets. `wandb.login()` reads the `WANDB_API_KEY` environment variable, so nothing is pasted or printed.

In [ ]:
import wandb
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

## 5. Run the matrix
One subprocess per job, so each run starts with a clean CUDA context. A run that already finished (recorded in `outputs/split_seeds/runs.json` inside the clone) is skipped, so the cell can be re-run after a disconnect. Add `--only split-v2 --smoke` for a one-minute check first.

In [ ]:
!cd rxnorm && python scripts/12_split_seeds.py train --from-artifact

## 6. Afterwards
Back on the local machine: `uv run scripts/12_split_seeds.py summarize` pulls the eight runs from W&B by group and writes `outputs/split_seeds/summary.{json,md}`. In the W&B project, filter the runs table by group `split-seeds` and pin `val/acc@1` and `test/acc@1`.